In [1]:
# Cell 1 — Environment audit
# This cell only records the execution environment.

from pathlib import Path
import sys
import platform
import gzip
import json

import pandas as pd

CURRENT_WORKING_DIR = Path.cwd().resolve()

print("Environment audit")
print("-" * 60)
print(f"Python version       : {sys.version.split()[0]}")
print(f"Python executable    : {sys.executable}")
print(f"Operating system     : {platform.platform()}")
print(f"Pandas version       : {pd.__version__}")
print(f"Current working dir  : {CURRENT_WORKING_DIR}")

Environment audit
------------------------------------------------------------
Python version       : 3.13.11
Python executable    : C:\Users\HP\AppData\Local\Programs\Python\Python313\python.exe
Operating system     : Windows-11-10.0.26200-SP0
Pandas version       : 2.3.3
Current working dir  : C:\Users\HP\Desktop\thesis_preprocessing\notebooks


In [2]:
# Cell 2 — Resolve and validate project paths
# This cell does not create, delete, or modify any files.

PROJECT_ROOT = CURRENT_WORKING_DIR.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
ORIGINAL_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REWORK_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed_rework_v2"

ORIGINAL_OUTPUT_DIR = PROJECT_ROOT / "outputs"
REWORK_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "rework_v2"

LANGUAGES = (
    "go",
    "java",
    "javascript",
    "php",
    "python",
    "ruby",
)

required_existing_paths = {
    "Project root": PROJECT_ROOT,
    "Raw data": RAW_DATA_DIR,
    "Original processed data": ORIGINAL_PROCESSED_DIR,
    "Original outputs": ORIGINAL_OUTPUT_DIR,
}

print("Project path audit")
print("-" * 80)

for label, path in required_existing_paths.items():
    print(f"{label:<28}: {path}")
    print(f"{'Exists':<28}: {path.exists()}")
    print(f"{'Is directory':<28}: {path.is_dir()}")
    print()

print("Expected raw JSONL directories")
print("-" * 80)

missing_language_paths = []

for language in LANGUAGES:
    jsonl_path = RAW_DATA_DIR / language / "final" / "jsonl"
    exists = jsonl_path.is_dir()

    print(f"{language:<12}: {jsonl_path} | exists={exists}")

    if not exists:
        missing_language_paths.append(language)

assert PROJECT_ROOT.name == "thesis_preprocessing", (
    f"Unexpected project root: {PROJECT_ROOT}"
)

assert RAW_DATA_DIR.is_dir(), (
    f"Raw-data directory was not found: {RAW_DATA_DIR}"
)

assert ORIGINAL_PROCESSED_DIR.is_dir(), (
    f"Original processed-data directory was not found: "
    f"{ORIGINAL_PROCESSED_DIR}"
)

assert ORIGINAL_OUTPUT_DIR.is_dir(), (
    f"Original output directory was not found: {ORIGINAL_OUTPUT_DIR}"
)

assert not missing_language_paths, (
    "Missing raw JSONL directories for: "
    + ", ".join(missing_language_paths)
)

print("\nPATH AUDIT PASSED")
print("No files or directories were modified.")

Project path audit
--------------------------------------------------------------------------------
Project root                : C:\Users\HP\Desktop\thesis_preprocessing
Exists                      : True
Is directory                : True

Raw data                    : C:\Users\HP\Desktop\thesis_preprocessing\data\raw
Exists                      : True
Is directory                : True

Original processed data     : C:\Users\HP\Desktop\thesis_preprocessing\data\processed
Exists                      : True
Is directory                : True

Original outputs            : C:\Users\HP\Desktop\thesis_preprocessing\outputs
Exists                      : True
Is directory                : True

Expected raw JSONL directories
--------------------------------------------------------------------------------
go          : C:\Users\HP\Desktop\thesis_preprocessing\data\raw\go\final\jsonl | exists=True
java        : C:\Users\HP\Desktop\thesis_preprocessing\data\raw\java\final\jsonl | exists=True


In [3]:
# Cell 3 — Inventory raw CodeSearchNet files
# Read-only: this cell does not open dataset records or modify files.

inventory_rows = []

for language in LANGUAGES:
    language_root = RAW_DATA_DIR / language / "final" / "jsonl"
    split_directories = sorted(
        path for path in language_root.iterdir()
        if path.is_dir()
    )

    if not split_directories:
        inventory_rows.append({
            "language": language,
            "split": None,
            "gzip_files": 0,
            "compressed_bytes": 0,
            "first_file": None,
            "last_file": None,
        })
        continue

    for split_directory in split_directories:
        gzip_files = sorted(split_directory.glob("*.jsonl.gz"))

        inventory_rows.append({
            "language": language,
            "split": split_directory.name,
            "gzip_files": len(gzip_files),
            "compressed_bytes": sum(
                file_path.stat().st_size
                for file_path in gzip_files
            ),
            "first_file": gzip_files[0].name if gzip_files else None,
            "last_file": gzip_files[-1].name if gzip_files else None,
        })

raw_file_inventory = pd.DataFrame(inventory_rows)

raw_file_inventory["compressed_mb"] = (
    raw_file_inventory["compressed_bytes"] / (1024 ** 2)
).round(2)

display_columns = [
    "language",
    "split",
    "gzip_files",
    "compressed_mb",
    "first_file",
    "last_file",
]

print("Raw CodeSearchNet file inventory")
print("-" * 100)
display(raw_file_inventory[display_columns])

print("\nSummary by language")
print("-" * 100)

language_inventory_summary = (
    raw_file_inventory
    .groupby("language", as_index=False)
    .agg(
        splits=("split", "nunique"),
        gzip_files=("gzip_files", "sum"),
        compressed_bytes=("compressed_bytes", "sum"),
    )
)

language_inventory_summary["compressed_gb"] = (
    language_inventory_summary["compressed_bytes"] / (1024 ** 3)
).round(3)

display(
    language_inventory_summary[
        ["language", "splits", "gzip_files", "compressed_gb"]
    ]
)

assert not raw_file_inventory.empty, "No raw-file inventory was produced."
assert raw_file_inventory["gzip_files"].sum() > 0, (
    "No .jsonl.gz files were found."
)

languages_without_files = (
    language_inventory_summary.loc[
        language_inventory_summary["gzip_files"] == 0,
        "language",
    ]
    .tolist()
)

assert not languages_without_files, (
    "No raw files were found for: "
    + ", ".join(languages_without_files)
)

print("\nRAW FILE INVENTORY PASSED")
print("No raw records were opened and no files were modified.")

Raw CodeSearchNet file inventory
----------------------------------------------------------------------------------------------------


,language,split,gzip_files,compressed_mb,first_file,last_file
0,go,test,1,4.33,go_test_0.jsonl.gz,go_test_0.jsonl.gz
1,go,train,11,93.53,go_train_0.jsonl.gz,go_train_9.jsonl.gz
2,go,valid,1,3.37,go_valid_0.jsonl.gz,go_valid_0.jsonl.gz
3,java,test,1,10.33,java_test_0.jsonl.gz,java_test_0.jsonl.gz
4,java,train,16,167.69,java_train_0.jsonl.gz,java_train_9.jsonl.gz
5,java,valid,1,5.27,java_valid_0.jsonl.gz,java_valid_0.jsonl.gz
6,javascript,test,1,3.51,javascript_test_0.jsonl.gz,javascript_test_0.jsonl.gz
7,javascript,train,5,69.32,javascript_train_0.jsonl.gz,javascript_train_4.jsonl.gz
8,javascript,valid,1,4.36,javascript_valid_0.jsonl.gz,javascript_valid_0.jsonl.gz
9,php,test,1,9.93,php_test_0.jsonl.gz,php_test_0.jsonl.gz



Summary by language
----------------------------------------------------------------------------------------------------


,language,splits,gzip_files,compressed_gb
0,go,3,13,0.099
1,java,3,18,0.179
2,javascript,3,7,0.075
3,php,3,20,0.207
4,python,3,16,0.205
5,ruby,3,4,0.019



RAW FILE INVENTORY PASSED
No raw records were opened and no files were modified.


In [4]:
# Cell 4 — Audit the raw CodeSearchNet JSON schema
# Read-only: opens one parseable record from every compressed shard.
# It does not store raw code or modify any file.

def read_first_parseable_record(file_path: Path):
    """Return the first JSON object found in one gzip JSONL shard."""
    json_failures = 0

    with gzip.open(
        file_path,
        mode="rt",
        encoding="utf-8",
        errors="strict",
    ) as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                json_failures += 1
                continue

            if isinstance(record, dict):
                return record, line_number, json_failures

    return None, None, json_failures


schema_rows = []
sampled_records = []

for language in LANGUAGES:
    language_root = RAW_DATA_DIR / language / "final" / "jsonl"

    for split in ("train", "valid", "test"):
        split_dir = language_root / split
        shard_paths = sorted(split_dir.glob("*.jsonl.gz"))

        for shard_path in shard_paths:
            record, line_number, json_failures = (
                read_first_parseable_record(shard_path)
            )

            schema_rows.append({
                "language": language,
                "split": split,
                "file": shard_path.name,
                "record_found": record is not None,
                "line_number": line_number,
                "json_failures_before_record": json_failures,
                "key_count": len(record) if record is not None else 0,
            })

            if record is not None:
                sampled_records.append({
                    "language": language,
                    "split": split,
                    "file": shard_path.name,
                    "record": record,
                })


shard_schema_audit = pd.DataFrame(schema_rows)

print("Shard-level schema audit")
print("-" * 100)

display(
    shard_schema_audit.groupby(
        ["language", "split"],
        as_index=False,
    ).agg(
        shards=("file", "count"),
        records_found=("record_found", "sum"),
        minimum_key_count=("key_count", "min"),
        maximum_key_count=("key_count", "max"),
        json_failures=("json_failures_before_record", "sum"),
    )
)


all_key_sets = [
    set(item["record"].keys())
    for item in sampled_records
]

union_of_keys = set().union(*all_key_sets)
intersection_of_keys = set.intersection(*all_key_sets)

print("\nKeys present in every sampled shard")
print("-" * 100)
print(sorted(intersection_of_keys))

print("\nKeys found in at least one sampled shard")
print("-" * 100)
print(sorted(union_of_keys))


field_rows = []

for field_name in sorted(union_of_keys):
    observed_values = [
        item["record"].get(field_name)
        for item in sampled_records
        if field_name in item["record"]
    ]

    observed_types = sorted({
        type(value).__name__
        for value in observed_values
    })

    non_null_count = sum(
        value is not None
        for value in observed_values
    )

    field_rows.append({
        "field": field_name,
        "present_in_shards": len(observed_values),
        "non_null_in_shards": non_null_count,
        "observed_types": ", ".join(observed_types),
    })


field_presence_audit = pd.DataFrame(field_rows)

print("\nField presence and observed data types")
print("-" * 100)
display(field_presence_audit)


total_shards = len(shard_schema_audit)
successful_shards = int(
    shard_schema_audit["record_found"].sum()
)

assert total_shards > 0, "No compressed shards were audited."

assert successful_shards == total_shards, (
    f"Only {successful_shards} of {total_shards} shards "
    "contained a parseable JSON object."
)

assert sampled_records, "No sample records were collected."

print("\nRAW SCHEMA AUDIT PASSED")
print(f"Audited shards: {total_shards}")
print("No raw code or comments were printed.")
print("No files were modified.")

Shard-level schema audit
----------------------------------------------------------------------------------------------------


,language,split,shards,records_found,minimum_key_count,maximum_key_count,json_failures
0,go,test,1,1,12,12,0
1,go,train,11,11,12,12,0
2,go,valid,1,1,12,12,0
3,java,test,1,1,12,12,0
4,java,train,16,16,12,12,0
5,java,valid,1,1,12,12,0
6,javascript,test,1,1,12,12,0
7,javascript,train,5,5,12,12,0
8,javascript,valid,1,1,12,12,0
9,php,test,1,1,12,12,0



Keys present in every sampled shard
----------------------------------------------------------------------------------------------------
['code', 'code_tokens', 'docstring', 'docstring_tokens', 'func_name', 'language', 'original_string', 'partition', 'path', 'repo', 'sha', 'url']

Keys found in at least one sampled shard
----------------------------------------------------------------------------------------------------
['code', 'code_tokens', 'docstring', 'docstring_tokens', 'func_name', 'language', 'original_string', 'partition', 'path', 'repo', 'sha', 'url']

Field presence and observed data types
----------------------------------------------------------------------------------------------------


,field,present_in_shards,non_null_in_shards,observed_types
0,code,78,78,str
1,code_tokens,78,78,list
2,docstring,78,78,str
3,docstring_tokens,78,78,list
4,func_name,78,78,str
5,language,78,78,str
6,original_string,78,78,str
7,partition,78,78,str
8,path,78,78,str
9,repo,78,78,str



RAW SCHEMA AUDIT PASSED
Audited shards: 78
No raw code or comments were printed.
No files were modified.


In [6]:
# Cell 5B — Complete the full-population audit report
# Uses the completed Cell 5 scan already held in memory.
# Does not reread raw files and does not write anything.

EXPECTED_AUDIT_COLUMNS = (
    "shards",
    "jsonl_lines",
    "blank_jsonl_lines",
    "json_decode_errors",
    "non_dictionary_records",
    "parsed_records",
    "records_with_missing_fields",
    "records_with_wrong_types",
    "empty_code",
    "empty_docstring",
    "empty_repo",
    "empty_path",
    "empty_func_name",
    "empty_sha",
    "language_mismatches",
    "partition_mismatches",
    "legacy_boilerplate_matches",
    "legacy_eligible_records",
)

# Counter omitted categories whose count was always zero.
# Add those missing columns explicitly.
for column in EXPECTED_AUDIT_COLUMNS:
    if column not in population_audit_by_split.columns:
        population_audit_by_split[column] = 0

population_audit_by_split = (
    population_audit_by_split[
        ["language", "split", *EXPECTED_AUDIT_COLUMNS]
    ]
    .copy()
)

population_audit_by_split[
    list(EXPECTED_AUDIT_COLUMNS)
] = (
    population_audit_by_split[
        list(EXPECTED_AUDIT_COLUMNS)
    ]
    .fillna(0)
    .astype("int64")
)


population_audit_by_language = (
    population_audit_by_split
    .groupby("language", as_index=False)[
        list(EXPECTED_AUDIT_COLUMNS)
    ]
    .sum()
)

population_audit_by_language[
    "legacy_retention_percent"
] = (
    100
    * population_audit_by_language["legacy_eligible_records"]
    / population_audit_by_language["parsed_records"]
).round(2)


print("Population audit by language")
print("-" * 120)

display(
    population_audit_by_language[
        [
            "language",
            "shards",
            "parsed_records",
            "legacy_eligible_records",
            "legacy_retention_percent",
            "legacy_boilerplate_matches",
            "json_decode_errors",
            "records_with_missing_fields",
            "records_with_wrong_types",
            "empty_code",
            "empty_docstring",
            "empty_repo",
            "empty_path",
            "empty_func_name",
            "empty_sha",
            "language_mismatches",
            "partition_mismatches",
        ]
    ]
)


field_issue_rows = []

for (language, split), issues in field_issue_counts.items():
    for issue, count in issues.items():
        field_issue_rows.append({
            "language": language,
            "split": split,
            "issue": issue,
            "count": count,
        })

field_issue_audit = pd.DataFrame(
    field_issue_rows,
    columns=["language", "split", "issue", "count"],
)

print("\nDetailed field issues")
print("-" * 120)

if field_issue_audit.empty:
    print("No missing-field or wrong-type issues detected.")
else:
    display(
        field_issue_audit.sort_values(
            ["language", "split", "issue"]
        )
    )


boilerplate_rows = []

for language, term_counts in boilerplate_term_counts.items():
    for term, count in term_counts.items():
        boilerplate_rows.append({
            "language": language,
            "term": term,
            "matches": count,
        })

legacy_boilerplate_audit = pd.DataFrame(
    boilerplate_rows,
    columns=["language", "term", "matches"],
)

print("\nLegacy boilerplate-term matches")
print("-" * 120)

if legacy_boilerplate_audit.empty:
    print("No legacy boilerplate terms were detected.")
else:
    display(
        legacy_boilerplate_audit.sort_values(
            ["language", "matches"],
            ascending=[True, False],
        )
    )


total_parsed_records = int(
    population_audit_by_split["parsed_records"].sum()
)

total_jsonl_lines = int(
    population_audit_by_split["jsonl_lines"].sum()
)

assert total_parsed_records > 0

assert total_jsonl_lines >= total_parsed_records

assert (
    population_audit_by_split["json_decode_errors"].sum()
    == 0
), "Unexpected JSON decoding errors were detected."

assert (
    population_audit_by_split[
        "records_with_missing_fields"
    ].sum()
    == 0
), "Records with missing fields were detected."

assert (
    population_audit_by_split[
        "records_with_wrong_types"
    ].sum()
    == 0
), "Records with incorrect field types were detected."


print("\nFULL RAW-POPULATION AUDIT COMPLETED")
print(f"Total parsed records: {total_parsed_records:,}")
print("No records were sampled.")
print("No files were written or modified.")

Population audit by language
------------------------------------------------------------------------------------------------------------------------


,language,shards,parsed_records,legacy_eligible_records,legacy_retention_percent,legacy_boilerplate_matches,json_decode_errors,records_with_missing_fields,records_with_wrong_types,empty_code,empty_docstring,empty_repo,empty_path,empty_func_name,empty_sha,language_mismatches,partition_mismatches
0,go,13,346365,336833,97.25,3247,0,0,0,0,0,0,0,0,0,0,0
1,java,18,496688,482408,97.12,4000,0,0,0,0,0,0,0,0,0,0,0
2,javascript,7,138625,131763,95.05,1598,0,0,0,0,0,0,0,47196,0,0,0
3,php,20,578118,555541,96.09,10348,0,0,0,0,0,0,0,0,0,0,0
4,python,16,457461,443500,96.95,4242,0,0,0,0,0,0,0,0,0,0,0
5,ruby,4,53279,52238,98.05,895,0,0,0,0,0,0,0,0,0,0,0



Detailed field issues
------------------------------------------------------------------------------------------------------------------------
No missing-field or wrong-type issues detected.

Legacy boilerplate-term matches
------------------------------------------------------------------------------------------------------------------------


,language,term,matches
0,go,author,3007
1,go,license,263
3,go,copyright,66
2,go,auto-generated,35
4,go,all rights reserved,14
5,java,author,3525
6,java,license,401
8,java,auto-generated,79
7,java,copyright,56
9,java,all rights reserved,3



FULL RAW-POPULATION AUDIT COMPLETED
Total parsed records: 2,070,536
No records were sampled.
No files were written or modified.


In [7]:
# Cell 6 — Summarise code/comment length distributions
# Uses the histograms already created in Cell 5.
# No files are read or written.

QUANTILES = (
    0.00,
    0.01,
    0.05,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99,
    1.00,
)


def counter_quantiles(counter: Counter, quantiles=QUANTILES):
    """
    Calculate exact empirical quantiles from a frequency Counter
    without expanding millions of observations into a list.
    """
    total = sum(counter.values())

    if total == 0:
        return {
            quantile: None
            for quantile in quantiles
        }

    sorted_items = sorted(counter.items())
    results = {}

    cumulative = 0
    target_positions = {
        quantile: max(
            1,
            int(round(quantile * (total - 1))) + 1,
        )
        for quantile in quantiles
    }

    quantile_index = 0
    ordered_quantiles = sorted(quantiles)

    for value, frequency in sorted_items:
        cumulative += frequency

        while (
            quantile_index < len(ordered_quantiles)
            and cumulative
            >= target_positions[ordered_quantiles[quantile_index]]
        ):
            quantile = ordered_quantiles[quantile_index]
            results[quantile] = value
            quantile_index += 1

        if quantile_index == len(ordered_quantiles):
            break

    return results


length_summary_rows = []

for metric in LENGTH_METRICS:
    for language in LANGUAGES:
        histogram = length_histograms[metric][language]
        total_records = sum(histogram.values())
        quantile_values = counter_quantiles(histogram)

        length_summary_rows.append({
            "metric": metric,
            "language": language,
            "records": total_records,
            "minimum": quantile_values[0.00],
            "p01": quantile_values[0.01],
            "p05": quantile_values[0.05],
            "p25": quantile_values[0.25],
            "median": quantile_values[0.50],
            "p75": quantile_values[0.75],
            "p90": quantile_values[0.90],
            "p95": quantile_values[0.95],
            "p99": quantile_values[0.99],
            "maximum": quantile_values[1.00],
        })


length_distribution_summary = pd.DataFrame(length_summary_rows)

print("Length-distribution summary")
print("-" * 120)

for metric in LENGTH_METRICS:
    print(f"\nMetric: {metric}")
    display(
        length_distribution_summary.loc[
            length_distribution_summary["metric"] == metric,
            [
                "language",
                "records",
                "minimum",
                "p01",
                "p05",
                "p25",
                "median",
                "p75",
                "p90",
                "p95",
                "p99",
                "maximum",
            ],
        ].reset_index(drop=True)
    )


assert len(length_distribution_summary) == (
    len(LENGTH_METRICS) * len(LANGUAGES)
)

assert (
    length_distribution_summary["records"] > 0
).all(), "At least one language/metric combination has no data."

print("\nLENGTH-DISTRIBUTION SUMMARY COMPLETED")
print("No thresholds were selected.")
print("No files were read or modified.")

Length-distribution summary
------------------------------------------------------------------------------------------------------------------------

Metric: code_tokens


,language,records,minimum,p01,p05,p25,median,p75,p90,p95,p99,maximum
0,go,346365,21,24,25,34,61,116,217,319,671,188228
1,java,496688,20,24,28,42,66,121,224,331,710,68278
2,javascript,138625,22,29,35,56,91,165,301,448,1062,671620
3,php,578118,24,32,37,53,81,140,243,347,726,45392
4,python,457461,19,24,28,43,72,132,237,341,714,28410
5,ruby,53279,15,23,26,34,48,77,125,174,336,45331



Metric: comment_tokens


,language,records,minimum,p01,p05,p25,median,p75,p90,p95,p99,maximum
0,go,346365,1,2,5,8,12,23,49,92,181,3916
1,java,496688,1,1,1,7,11,21,39,61,134,3439
2,javascript,138625,1,1,1,5,10,18,33,47,97,2104
3,php,578118,1,1,1,4,7,11,17,24,48,1097
4,python,457461,1,2,4,7,10,17,33,48,98,1971
5,ruby,53279,1,1,3,7,11,20,36,49,91,2125



Metric: combined_tokens


...


Metric: code_lines


,language,records,minimum,p01,p05,p25,median,p75,p90,p95,p99,maximum
0,go,346365,3,3,3,4,8,17,34,50,105,30635
1,java,496688,3,3,3,6,10,19,36,54,120,11326
2,javascript,138625,3,3,5,9,16,29,54,81,198,66742
3,php,578118,3,4,5,9,13,22,39,55,112,9629
4,python,457461,3,4,6,11,18,31,54,75,148,2773
5,ruby,53279,3,3,3,5,9,15,25,35,68,684



Metric: comment_lines


,language,records,minimum,p01,p05,p25,median,p75,p90,p95,p99,maximum
0,go,346365,1,1,1,1,1,2,5,9,26,861
1,java,496688,1,1,1,2,4,7,11,14,28,513
2,javascript,138625,1,1,1,1,3,6,10,15,34,632
3,php,578118,1,1,1,3,4,6,9,11,20,1329
4,python,457461,1,1,1,1,4,9,17,25,54,1444
5,ruby,53279,1,1,1,1,4,9,14,20,46,819



LENGTH-DISTRIBUTION SUMMARY COMPLETED
No thresholds were selected.
No files were read or modified.


In [8]:
# Cell 7 — Audit candidate lower and upper token bounds
# Uses Cell 5 histograms only.
# Does not reread raw files, filter records, or write outputs.

CODE_UPPER_BOUNDS = (128, 256, 384, 512, 768, 1024, 2048)
COMBINED_UPPER_BOUNDS = (128, 256, 384, 512, 768, 1024, 2048)

COMMENT_LOWER_BOUNDS = (1, 3, 5, 10)
COMMENT_UPPER_BOUNDS = (32, 64, 128, 256, 512)


def count_at_most(counter: Counter, upper_bound: int) -> int:
    return sum(
        frequency
        for value, frequency in counter.items()
        if value <= upper_bound
    )


def count_at_least(counter: Counter, lower_bound: int) -> int:
    return sum(
        frequency
        for value, frequency in counter.items()
        if value >= lower_bound
    )


def build_upper_bound_table(metric: str, bounds: tuple[int, ...]):
    rows = []

    for language in LANGUAGES:
        histogram = length_histograms[metric][language]
        total = sum(histogram.values())

        for bound in bounds:
            retained = count_at_most(histogram, bound)

            rows.append({
                "language": language,
                "upper_bound": bound,
                "retained_records": retained,
                "excluded_records": total - retained,
                "retained_percent": round(
                    100 * retained / total,
                    2,
                ),
            })

    return pd.DataFrame(rows)


def build_lower_bound_table(metric: str, bounds: tuple[int, ...]):
    rows = []

    for language in LANGUAGES:
        histogram = length_histograms[metric][language]
        total = sum(histogram.values())

        for bound in bounds:
            retained = count_at_least(histogram, bound)

            rows.append({
                "language": language,
                "lower_bound": bound,
                "retained_records": retained,
                "excluded_records": total - retained,
                "retained_percent": round(
                    100 * retained / total,
                    2,
                ),
            })

    return pd.DataFrame(rows)


code_upper_bound_audit = build_upper_bound_table(
    "code_tokens",
    CODE_UPPER_BOUNDS,
)

combined_upper_bound_audit = build_upper_bound_table(
    "combined_tokens",
    COMBINED_UPPER_BOUNDS,
)

comment_lower_bound_audit = build_lower_bound_table(
    "comment_tokens",
    COMMENT_LOWER_BOUNDS,
)

comment_upper_bound_audit = build_upper_bound_table(
    "comment_tokens",
    COMMENT_UPPER_BOUNDS,
)


def display_retention_pivot(
    dataframe: pd.DataFrame,
    bound_column: str,
    title: str,
):
    pivot = dataframe.pivot(
        index="language",
        columns=bound_column,
        values="retained_percent",
    )

    print(f"\n{title}")
    print("-" * 100)
    display(pivot)


display_retention_pivot(
    code_upper_bound_audit,
    "upper_bound",
    "Percentage retained under candidate CODE-token upper bounds",
)

display_retention_pivot(
    combined_upper_bound_audit,
    "upper_bound",
    "Percentage retained under candidate COMBINED-token upper bounds",
)

display_retention_pivot(
    comment_lower_bound_audit,
    "lower_bound",
    "Percentage retained under candidate COMMENT-token lower bounds",
)

display_retention_pivot(
    comment_upper_bound_audit,
    "upper_bound",
    "Percentage retained under candidate COMMENT-token upper bounds",
)


assert len(code_upper_bound_audit) == (
    len(LANGUAGES) * len(CODE_UPPER_BOUNDS)
)

assert len(combined_upper_bound_audit) == (
    len(LANGUAGES) * len(COMBINED_UPPER_BOUNDS)
)

assert len(comment_lower_bound_audit) == (
    len(LANGUAGES) * len(COMMENT_LOWER_BOUNDS)
)

print("\nCANDIDATE-BOUND COVERAGE AUDIT COMPLETED")
print("These are marginal retention rates, not final combined filters.")
print("No threshold was selected.")
print("No files were read or modified.")


Percentage retained under candidate CODE-token upper bounds
----------------------------------------------------------------------------------------------------


upper_bound,128,256,384,512,768,1024,2048
language,,,,,,,
go,77.96,92.54,96.55,98.16,99.25,99.61,99.93
java,76.95,92.09,96.27,97.91,99.17,99.57,99.92
javascript,65.55,86.94,93.38,96.10,98.18,98.93,99.69
php,71.84,90.93,95.93,97.81,99.12,99.55,99.90
python,74.12,91.36,96.08,97.85,99.15,99.58,99.93
ruby,90.49,97.96,99.32,99.71,99.91,99.97,99.99



Percentage retained under candidate COMBINED-token upper bounds
----------------------------------------------------------------------------------------------------


upper_bound,128,256,384,512,768,1024,2048
language,,,,,,,
go,68.27,89.10,95.69,97.76,99.13,99.56,99.93
java,70.16,90.45,95.67,97.64,99.09,99.54,99.91
javascript,59.56,85.21,92.68,95.76,98.08,98.88,99.68
php,68.62,90.21,95.67,97.70,99.09,99.54,99.90
python,68.71,89.77,95.50,97.57,99.06,99.53,99.92
ruby,85.52,97.29,99.13,99.62,99.89,99.95,99.99



Percentage retained under candidate COMMENT-token lower bounds
----------------------------------------------------------------------------------------------------


lower_bound,1,3,5,10
language,,,,
go,100.0,98.53,95.48,60.84
java,100.0,91.74,84.73,58.60
javascript,100.0,90.54,78.91,50.72
php,100.0,84.18,67.22,30.21
python,100.0,98.43,91.22,56.60
ruby,100.0,96.56,88.22,58.04



Percentage retained under candidate COMMENT-token upper bounds
----------------------------------------------------------------------------------------------------


upper_bound,32,64,128,256,512
language,,,,,
go,83.48,92.49,96.96,99.57,99.92
java,86.47,95.47,98.92,99.77,99.96
javascript,89.93,97.48,99.47,99.89,99.97
php,97.34,99.57,99.94,99.99,100.00
python,89.97,97.43,99.44,99.88,99.98
ruby,88.25,97.35,99.62,99.92,99.98



CANDIDATE-BOUND COVERAGE AUDIT COMPLETED
These are marginal retention rates, not final combined filters.
No threshold was selected.
No files were read or modified.


In [9]:
# Cell 8 — Audit code-line and comment-line distributions
# Uses the histograms already created in Cell 5.
# Does not reread or modify files.

CODE_LINE_LOWER_BOUNDS = (1, 2, 3, 5, 10)
COMMENT_LINE_LOWER_BOUNDS = (1, 2, 3, 5)


def build_line_lower_bound_table(
    metric: str,
    bounds: tuple[int, ...],
) -> pd.DataFrame:
    rows = []

    for language in LANGUAGES:
        histogram = length_histograms[metric][language]
        total = sum(histogram.values())

        for lower_bound in bounds:
            retained = sum(
                frequency
                for value, frequency in histogram.items()
                if value >= lower_bound
            )

            rows.append({
                "language": language,
                "lower_bound": lower_bound,
                "retained_records": retained,
                "excluded_records": total - retained,
                "retained_percent": round(
                    100 * retained / total,
                    2,
                ),
            })

    return pd.DataFrame(rows)


print("Code-line distribution summary")
print("-" * 100)

display(
    length_distribution_summary.loc[
        length_distribution_summary["metric"] == "code_lines",
        [
            "language",
            "records",
            "minimum",
            "p01",
            "p05",
            "p25",
            "median",
            "p75",
            "p90",
            "p95",
            "p99",
            "maximum",
        ],
    ].reset_index(drop=True)
)


print("\nComment-line distribution summary")
print("-" * 100)

display(
    length_distribution_summary.loc[
        length_distribution_summary["metric"] == "comment_lines",
        [
            "language",
            "records",
            "minimum",
            "p01",
            "p05",
            "p25",
            "median",
            "p75",
            "p90",
            "p95",
            "p99",
            "maximum",
        ],
    ].reset_index(drop=True)
)


code_line_lower_bound_audit = build_line_lower_bound_table(
    "code_lines",
    CODE_LINE_LOWER_BOUNDS,
)

comment_line_lower_bound_audit = build_line_lower_bound_table(
    "comment_lines",
    COMMENT_LINE_LOWER_BOUNDS,
)


print("\nPercentage retained under candidate CODE-line lower bounds")
print("-" * 100)

display(
    code_line_lower_bound_audit.pivot(
        index="language",
        columns="lower_bound",
        values="retained_percent",
    )
)


print("\nPercentage retained under candidate COMMENT-line lower bounds")
print("-" * 100)

display(
    comment_line_lower_bound_audit.pivot(
        index="language",
        columns="lower_bound",
        values="retained_percent",
    )
)


assert len(code_line_lower_bound_audit) == (
    len(LANGUAGES) * len(CODE_LINE_LOWER_BOUNDS)
)

assert len(comment_line_lower_bound_audit) == (
    len(LANGUAGES) * len(COMMENT_LINE_LOWER_BOUNDS)
)

print("\nLINE-COUNT AUDIT COMPLETED")
print("No line-count threshold was selected.")
print("No files were read or modified.")

Code-line distribution summary
----------------------------------------------------------------------------------------------------


,language,records,minimum,p01,p05,p25,median,p75,p90,p95,p99,maximum
0,go,346365,3,3,3,4,8,17,34,50,105,30635
1,java,496688,3,3,3,6,10,19,36,54,120,11326
2,javascript,138625,3,3,5,9,16,29,54,81,198,66742
3,php,578118,3,4,5,9,13,22,39,55,112,9629
4,python,457461,3,4,6,11,18,31,54,75,148,2773
5,ruby,53279,3,3,3,5,9,15,25,35,68,684



Comment-line distribution summary
----------------------------------------------------------------------------------------------------


,language,records,minimum,p01,p05,p25,median,p75,p90,p95,p99,maximum
0,go,346365,1,1,1,1,1,2,5,9,26,861
1,java,496688,1,1,1,2,4,7,11,14,28,513
2,javascript,138625,1,1,1,1,3,6,10,15,34,632
3,php,578118,1,1,1,3,4,6,9,11,20,1329
4,python,457461,1,1,1,1,4,9,17,25,54,1444
5,ruby,53279,1,1,1,1,4,9,14,20,46,819



Percentage retained under candidate CODE-line lower bounds
----------------------------------------------------------------------------------------------------


lower_bound,1,2,3,5,10
language,,,,,
go,100.0,100.0,100.0,71.60,44.77
java,100.0,100.0,100.0,85.66,53.19
javascript,100.0,100.0,100.0,96.25,73.74
php,100.0,100.0,100.0,95.91,69.67
python,100.0,100.0,100.0,97.53,81.06
ruby,100.0,100.0,100.0,82.00,48.29



Percentage retained under candidate COMMENT-line lower bounds
----------------------------------------------------------------------------------------------------


lower_bound,1,2,3,5
language,,,,
go,100.0,38.24,21.62,10.72
java,100.0,78.72,72.06,47.52
javascript,100.0,59.80,53.38,34.70
php,100.0,86.20,82.72,49.73
python,100.0,66.23,58.27,43.92
ruby,100.0,74.17,62.68,45.98



LINE-COUNT AUDIT COMPLETED
No line-count threshold was selected.
No files were read or modified.


In [10]:

# Creates only new generated directories and an empty SQLite database.
# It does not read raw records or alter legacy outputs.

import sqlite3

REWORK_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REWORK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_DB_PATH = (
    REWORK_PROCESSED_DIR / "01_candidate_population_index.sqlite"
)

connection = sqlite3.connect(CANDIDATE_DB_PATH)

try:
    connection.execute("""
        CREATE TABLE IF NOT EXISTS build_metadata (
            key TEXT PRIMARY KEY,
            value TEXT NOT NULL
        )
    """)

    connection.execute("""
        CREATE TABLE IF NOT EXISTS candidate_records (
            record_id INTEGER PRIMARY KEY AUTOINCREMENT,

            family_id TEXT NOT NULL UNIQUE,

            language TEXT NOT NULL,
            original_split TEXT NOT NULL,

            repository TEXT NOT NULL,
            file_path TEXT NOT NULL,
            function_name TEXT,
            commit_sha TEXT NOT NULL,
            source_url TEXT NOT NULL,

            shard_relative_path TEXT NOT NULL,
            shard_line_number INTEGER NOT NULL,

            code_token_count INTEGER NOT NULL,
            comment_token_count INTEGER NOT NULL,
            combined_token_count INTEGER NOT NULL,

            code_line_count INTEGER NOT NULL,
            comment_line_count INTEGER NOT NULL,

            exact_code_hash TEXT NOT NULL,
            exact_comment_hash TEXT NOT NULL,
            exact_pair_hash TEXT NOT NULL,

            legacy_boilerplate_match INTEGER NOT NULL
                CHECK (legacy_boilerplate_match IN (0, 1)),

            legacy_code_token_pass INTEGER NOT NULL
                CHECK (legacy_code_token_pass IN (0, 1))
        )
    """)

    connection.execute("""
        CREATE INDEX IF NOT EXISTS idx_candidate_language
        ON candidate_records(language)
    """)

    connection.execute("""
        CREATE INDEX IF NOT EXISTS idx_candidate_repository
        ON candidate_records(language, repository)
    """)

    connection.execute("""
        CREATE INDEX IF NOT EXISTS idx_candidate_split
        ON candidate_records(original_split)
    """)

    connection.execute("""
        CREATE INDEX IF NOT EXISTS idx_candidate_code_hash
        ON candidate_records(exact_code_hash)
    """)

    connection.execute("""
        CREATE INDEX IF NOT EXISTS idx_candidate_comment_hash
        ON candidate_records(exact_comment_hash)
    """)

    connection.execute("""
        CREATE INDEX IF NOT EXISTS idx_candidate_pair_hash
        ON candidate_records(exact_pair_hash)
    """)

    metadata = {
        "schema_version": "1",
        "build_status": "initialized",
        "raw_record_count_expected": str(
            int(population_audit_by_split["parsed_records"].sum())
        ),
        "languages": ",".join(LANGUAGES),
        "source_dataset": "CodeSearchNet",
    }

    connection.executemany(
        """
        INSERT INTO build_metadata(key, value)
        VALUES (?, ?)
        ON CONFLICT(key) DO UPDATE SET value = excluded.value
        """,
        metadata.items(),
    )

    connection.commit()

    tables = pd.read_sql_query(
        """
        SELECT name, type
        FROM sqlite_master
        WHERE type IN ('table', 'index')
        ORDER BY type, name
        """,
        connection,
    )

    metadata_check = pd.read_sql_query(
        """
        SELECT key, value
        FROM build_metadata
        ORDER BY key
        """,
        connection,
    )

    existing_record_count = connection.execute(
        "SELECT COUNT(*) FROM candidate_records"
    ).fetchone()[0]

finally:
    connection.close()


print("Candidate-index database initialised")
print("-" * 90)
print(f"Database path : {CANDIDATE_DB_PATH}")
print(f"Exists        : {CANDIDATE_DB_PATH.exists()}")
print(f"Size in bytes : {CANDIDATE_DB_PATH.stat().st_size:,}")
print(f"Record count  : {existing_record_count:,}")

print("\nDatabase metadata")
display(metadata_check)

print("\nDatabase objects")
display(tables)

assert CANDIDATE_DB_PATH.exists()
assert existing_record_count == 0
assert metadata_check.loc[
    metadata_check["key"] == "build_status",
    "value",
].iloc[0] == "initialized"

print("\nDATABASE INITIALISATION PASSED")
print("No raw records were read.")
print("No legacy files were changed.")

Candidate-index database initialised
------------------------------------------------------------------------------------------
Database path : C:\Users\HP\Desktop\thesis_preprocessing\data\processed_rework_v2\01_candidate_population_index.sqlite
Exists        : True
Size in bytes : 49,152
Record count  : 0

Database metadata


,key,value
0,build_status,initialized
1,languages,"go,java,javascript,php,python,ruby"
2,raw_record_count_expected,2070536
3,schema_version,1
4,source_dataset,CodeSearchNet



Database objects


,name,type
0,idx_candidate_code_hash,index
1,idx_candidate_comment_hash,index
2,idx_candidate_language,index
3,idx_candidate_pair_hash,index
4,idx_candidate_repository,index
5,idx_candidate_split,index
6,sqlite_autoindex_build_metadata_1,index
7,sqlite_autoindex_candidate_records_1,index
8,build_metadata,table
9,candidate_records,table



DATABASE INITIALISATION PASSED
No raw records were read.
No legacy files were changed.


In [11]:
import hashlib
import time
from datetime import datetime, timezone

BATCH_SIZE = 5_000

connection = sqlite3.connect(
    CANDIDATE_DB_PATH,
    timeout=60,
)

try:
    connection.execute("PRAGMA journal_mode = WAL")
    connection.execute("PRAGMA synchronous = NORMAL")
    connection.execute("PRAGMA temp_store = MEMORY")

    connection.execute("""
        CREATE TABLE IF NOT EXISTS processed_shards (
            shard_relative_path TEXT PRIMARY KEY,
            language TEXT NOT NULL,
            original_split TEXT NOT NULL,
            source_size_bytes INTEGER NOT NULL,
            source_mtime_ns INTEGER NOT NULL,
            inserted_records INTEGER NOT NULL,
            processed_at_utc TEXT NOT NULL
        )
    """)

    existing_columns = {
        row[1]
        for row in connection.execute(
            "PRAGMA table_info(candidate_records)"
        ).fetchall()
    }

    additional_columns = {
        "code_char_count": "INTEGER",
        "comment_char_count": "INTEGER",
        "nonempty_code_line_count": "INTEGER",
        "nonempty_comment_line_count": "INTEGER",
        "raw_record_hash": "TEXT",
    }

    for column_name, column_type in additional_columns.items():
        if column_name not in existing_columns:
            connection.execute(
                f"""
                ALTER TABLE candidate_records
                ADD COLUMN {column_name} {column_type}
                """
            )

    connection.execute("""
        CREATE UNIQUE INDEX IF NOT EXISTS
        idx_candidate_source_location
        ON candidate_records(
            shard_relative_path,
            shard_line_number
        )
    """)

    connection.commit()

    processed_shard_rows = connection.execute("""
        SELECT
            shard_relative_path,
            source_size_bytes,
            source_mtime_ns,
            inserted_records
        FROM processed_shards
    """).fetchall()

    processed_shards = {
        row[0]: {
            "source_size_bytes": row[1],
            "source_mtime_ns": row[2],
            "inserted_records": row[3],
        }
        for row in processed_shard_rows
    }

    insert_sql = """
        INSERT INTO candidate_records (
            family_id,
            language,
            original_split,
            repository,
            file_path,
            function_name,
            commit_sha,
            source_url,
            shard_relative_path,
            shard_line_number,
            code_token_count,
            comment_token_count,
            combined_token_count,
            code_line_count,
            comment_line_count,
            exact_code_hash,
            exact_comment_hash,
            exact_pair_hash,
            legacy_boilerplate_match,
            legacy_code_token_pass,
            code_char_count,
            comment_char_count,
            nonempty_code_line_count,
            nonempty_comment_line_count,
            raw_record_hash
        )
        VALUES (
            ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
            ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
            ?, ?, ?, ?, ?
        )
    """

    build_started = time.perf_counter()
    newly_inserted_total = 0

    for language in LANGUAGES:
        language_root = RAW_DATA_DIR / language / "final" / "jsonl"

        for split in ("train", "valid", "test"):
            shard_paths = sorted(
                (language_root / split).glob("*.jsonl.gz")
            )

            for shard_path in shard_paths:
                shard_relative_path = (
                    shard_path
                    .relative_to(PROJECT_ROOT)
                    .as_posix()
                )

                shard_stat = shard_path.stat()

                if shard_relative_path in processed_shards:
                    previous = processed_shards[
                        shard_relative_path
                    ]

                    if (
                        previous["source_size_bytes"]
                        != shard_stat.st_size
                        or previous["source_mtime_ns"]
                        != shard_stat.st_mtime_ns
                    ):
                        raise RuntimeError(
                            "A previously processed raw shard changed: "
                            f"{shard_relative_path}"
                        )

                    print(
                        f"Skipped {shard_relative_path} | "
                        f"already indexed="
                        f"{previous['inserted_records']:,}"
                    )
                    continue

                shard_started = time.perf_counter()
                shard_inserted = 0
                batch = []

                with connection:
                    with gzip.open(
                        shard_path,
                        mode="rt",
                        encoding="utf-8",
                        errors="strict",
                    ) as handle:

                        for line_number, raw_line in enumerate(
                            handle,
                            start=1,
                        ):
                            if not raw_line.strip():
                                continue

                            record = json.loads(raw_line)

                            code = record["code"]
                            comment = record["docstring"]
                            code_tokens = record["code_tokens"]
                            comment_tokens = record[
                                "docstring_tokens"
                            ]

                            clean_code = code.strip()
                            clean_comment = comment.strip()

                            code_lines = (
                                clean_code.splitlines()
                                if clean_code
                                else []
                            )

                            comment_lines = (
                                clean_comment.splitlines()
                                if clean_comment
                                else []
                            )

                            code_hash = hashlib.sha256(
                                code.encode("utf-8")
                            ).hexdigest()

                            comment_hash = hashlib.sha256(
                                comment.encode("utf-8")
                            ).hexdigest()

                            pair_hash = hashlib.sha256(
                                (
                                    code_hash
                                    + "\x1f"
                                    + comment_hash
                                ).encode("utf-8")
                            ).hexdigest()

                            raw_record_hash = hashlib.sha256(
                                raw_line.rstrip(
                                    "\r\n"
                                ).encode("utf-8")
                            ).hexdigest()

                            family_identity = "\x1f".join([
                                language,
                                split,
                                shard_relative_path,
                                str(line_number),
                                record["sha"],
                                record["url"],
                            ])

                            family_id = hashlib.sha256(
                                family_identity.encode("utf-8")
                            ).hexdigest()

                            comment_lower = clean_comment.lower()

                            legacy_boilerplate_match = int(
                                any(
                                    term in comment_lower
                                    for term
                                    in LEGACY_BOILERPLATE_TERMS
                                )
                            )

                            code_token_count = len(code_tokens)
                            comment_token_count = len(
                                comment_tokens
                            )

                            batch.append((
                                family_id,
                                language,
                                split,
                                record["repo"],
                                record["path"],
                                record["func_name"],
                                record["sha"],
                                record["url"],
                                shard_relative_path,
                                line_number,
                                code_token_count,
                                comment_token_count,
                                (
                                    code_token_count
                                    + comment_token_count
                                ),
                                len(code_lines),
                                len(comment_lines),
                                code_hash,
                                comment_hash,
                                pair_hash,
                                legacy_boilerplate_match,
                                int(code_token_count <= 512),
                                len(code),
                                len(comment),
                                sum(
                                    bool(line.strip())
                                    for line in code_lines
                                ),
                                sum(
                                    bool(line.strip())
                                    for line in comment_lines
                                ),
                                raw_record_hash,
                            ))

                            if len(batch) >= BATCH_SIZE:
                                connection.executemany(
                                    insert_sql,
                                    batch,
                                )

                                shard_inserted += len(batch)
                                batch.clear()

                        if batch:
                            connection.executemany(
                                insert_sql,
                                batch,
                            )

                            shard_inserted += len(batch)
                            batch.clear()

                    connection.execute(
                        """
                        INSERT INTO processed_shards (
                            shard_relative_path,
                            language,
                            original_split,
                            source_size_bytes,
                            source_mtime_ns,
                            inserted_records,
                            processed_at_utc
                        )
                        VALUES (?, ?, ?, ?, ?, ?, ?)
                        """,
                        (
                            shard_relative_path,
                            language,
                            split,
                            shard_stat.st_size,
                            shard_stat.st_mtime_ns,
                            shard_inserted,
                            datetime.now(
                                timezone.utc
                            ).isoformat(),
                        ),
                    )

                newly_inserted_total += shard_inserted

                elapsed = time.perf_counter() - shard_started

                print(
                    f"Indexed {shard_relative_path} | "
                    f"records={shard_inserted:,} | "
                    f"seconds={elapsed:.1f}"
                )

    final_record_count = connection.execute(
        "SELECT COUNT(*) FROM candidate_records"
    ).fetchone()[0]

    final_shard_count = connection.execute(
        "SELECT COUNT(*) FROM processed_shards"
    ).fetchone()[0]

    expected_record_count = int(
        connection.execute(
            """
            SELECT value
            FROM build_metadata
            WHERE key = 'raw_record_count_expected'
            """
        ).fetchone()[0]
    )

    expected_shard_count = int(
        raw_file_inventory["gzip_files"].sum()
    )

    null_audit = pd.read_sql_query("""
        SELECT
            SUM(code_char_count IS NULL)
                AS null_code_char_count,
            SUM(comment_char_count IS NULL)
                AS null_comment_char_count,
            SUM(nonempty_code_line_count IS NULL)
                AS null_nonempty_code_lines,
            SUM(nonempty_comment_line_count IS NULL)
                AS null_nonempty_comment_lines,
            SUM(raw_record_hash IS NULL)
                AS null_raw_record_hash
        FROM candidate_records
    """, connection)

    duplicate_family_count = connection.execute("""
        SELECT COUNT(*)
        FROM (
            SELECT family_id
            FROM candidate_records
            GROUP BY family_id
            HAVING COUNT(*) > 1
        )
    """).fetchone()[0]

    if final_record_count == expected_record_count:
        connection.execute(
            """
            INSERT INTO build_metadata(key, value)
            VALUES ('build_status', 'complete')
            ON CONFLICT(key)
            DO UPDATE SET value = excluded.value
            """
        )

        connection.execute(
            """
            INSERT INTO build_metadata(key, value)
            VALUES ('completed_at_utc', ?)
            ON CONFLICT(key)
            DO UPDATE SET value = excluded.value
            """,
            (
                datetime.now(
                    timezone.utc
                ).isoformat(),
            ),
        )

        connection.commit()

finally:
    connection.close()

total_elapsed = time.perf_counter() - build_started

print("\nCandidate population index")
print("-" * 90)
print(f"New records inserted : {newly_inserted_total:,}")
print(f"Final record count   : {final_record_count:,}")
print(f"Expected records     : {expected_record_count:,}")
print(f"Processed shards     : {final_shard_count:,}")
print(f"Expected shards      : {expected_shard_count:,}")
print(f"Duplicate family IDs : {duplicate_family_count:,}")
print(f"Elapsed seconds      : {total_elapsed:.1f}")

display(null_audit)

assert final_record_count == expected_record_count
assert final_shard_count == expected_shard_count
assert duplicate_family_count == 0
assert int(null_audit.sum(axis=1).iloc[0]) == 0

print("\nCANDIDATE POPULATION INDEX COMPLETED")

Indexed data/raw/go/final/jsonl/train/go_train_0.jsonl.gz | records=30,000 | seconds=3.2
Indexed data/raw/go/final/jsonl/train/go_train_1.jsonl.gz | records=30,000 | seconds=4.3
Indexed data/raw/go/final/jsonl/train/go_train_10.jsonl.gz | records=17,832 | seconds=2.4
Indexed data/raw/go/final/jsonl/train/go_train_2.jsonl.gz | records=30,000 | seconds=3.9
Indexed data/raw/go/final/jsonl/train/go_train_3.jsonl.gz | records=30,000 | seconds=3.9
Indexed data/raw/go/final/jsonl/train/go_train_4.jsonl.gz | records=30,000 | seconds=4.4
Indexed data/raw/go/final/jsonl/train/go_train_5.jsonl.gz | records=30,000 | seconds=4.9
Indexed data/raw/go/final/jsonl/train/go_train_6.jsonl.gz | records=30,000 | seconds=4.9
Indexed data/raw/go/final/jsonl/train/go_train_7.jsonl.gz | records=30,000 | seconds=5.1
Indexed data/raw/go/final/jsonl/train/go_train_8.jsonl.gz | records=30,000 | seconds=5.4
Indexed data/raw/go/final/jsonl/train/go_train_9.jsonl.gz | records=30,000 | seconds=5.6
Indexed data/raw/go/

,null_code_char_count,null_comment_char_count,null_nonempty_code_lines,null_nonempty_comment_lines,null_raw_record_hash
0,0,0,0,0,0



CANDIDATE POPULATION INDEX COMPLETED


In [12]:
connection = sqlite3.connect(
    CANDIDATE_DB_PATH,
    timeout=120,
)

try:
    population_structure = pd.read_sql_query("""
        SELECT
            language,
            COUNT(*) AS records,
            COUNT(DISTINCT repository) AS repositories,
            COUNT(DISTINCT exact_pair_hash) AS unique_pairs,
            COUNT(DISTINCT exact_code_hash) AS unique_code,
            COUNT(DISTINCT exact_comment_hash) AS unique_comments
        FROM candidate_records
        GROUP BY language
        ORDER BY language
    """, connection)

    duplicate_summary = pd.read_sql_query("""
        WITH
        pair_groups AS (
            SELECT
                language,
                COUNT(*) AS group_size
            FROM candidate_records
            GROUP BY language, exact_pair_hash
            HAVING COUNT(*) > 1
        ),
        code_groups AS (
            SELECT
                language,
                COUNT(*) AS group_size
            FROM candidate_records
            GROUP BY language, exact_code_hash
            HAVING COUNT(*) > 1
        ),
        comment_groups AS (
            SELECT
                language,
                COUNT(*) AS group_size
            FROM candidate_records
            GROUP BY language, exact_comment_hash
            HAVING COUNT(*) > 1
        )
        SELECT
            language,
            'exact_pair' AS duplicate_type,
            COUNT(*) AS duplicate_groups,
            SUM(group_size) AS records_in_duplicate_groups,
            SUM(group_size - 1) AS removable_duplicate_records,
            MAX(group_size) AS largest_group
        FROM pair_groups
        GROUP BY language

        UNION ALL

        SELECT
            language,
            'exact_code',
            COUNT(*),
            SUM(group_size),
            SUM(group_size - 1),
            MAX(group_size)
        FROM code_groups
        GROUP BY language

        UNION ALL

        SELECT
            language,
            'exact_comment',
            COUNT(*),
            SUM(group_size),
            SUM(group_size - 1),
            MAX(group_size)
        FROM comment_groups
        GROUP BY language

        ORDER BY language, duplicate_type
    """, connection)

    cross_split_duplicates = pd.read_sql_query("""
        WITH duplicate_groups AS (
            SELECT
                language,
                'exact_pair' AS duplicate_type,
                exact_pair_hash AS duplicate_hash,
                COUNT(*) AS records,
                COUNT(DISTINCT original_split) AS split_count
            FROM candidate_records
            GROUP BY language, exact_pair_hash
            HAVING COUNT(DISTINCT original_split) > 1

            UNION ALL

            SELECT
                language,
                'exact_code',
                exact_code_hash,
                COUNT(*),
                COUNT(DISTINCT original_split)
            FROM candidate_records
            GROUP BY language, exact_code_hash
            HAVING COUNT(DISTINCT original_split) > 1

            UNION ALL

            SELECT
                language,
                'exact_comment',
                exact_comment_hash,
                COUNT(*),
                COUNT(DISTINCT original_split)
            FROM candidate_records
            GROUP BY language, exact_comment_hash
            HAVING COUNT(DISTINCT original_split) > 1
        )
        SELECT
            language,
            duplicate_type,
            COUNT(*) AS cross_split_groups,
            SUM(records) AS records_in_cross_split_groups,
            MAX(records) AS largest_cross_split_group
        FROM duplicate_groups
        GROUP BY language, duplicate_type
        ORDER BY language, duplicate_type
    """, connection)

    largest_pair_duplicates = pd.read_sql_query("""
        SELECT
            language,
            exact_pair_hash,
            COUNT(*) AS records,
            COUNT(DISTINCT repository) AS repositories,
            COUNT(DISTINCT original_split) AS splits
        FROM candidate_records
        GROUP BY language, exact_pair_hash
        HAVING COUNT(*) > 1
        ORDER BY records DESC, language
        LIMIT 20
    """, connection)

finally:
    connection.close()

print("Population structure")
display(population_structure)

print("\nExact duplicate summary")
display(duplicate_summary)

print("\nDuplicates crossing original dataset splits")
display(cross_split_duplicates)

print("\nLargest exact-pair duplicate groups")
display(largest_pair_duplicates)

assert population_structure["records"].sum() == 2_070_536

print("\nDUPLICATE AND REPOSITORY AUDIT COMPLETED")

Population structure


,language,records,repositories,unique_pairs,unique_code,unique_comments
0,go,346365,4246,346365,346365,278084
1,java,496688,4769,496688,496688,423834
2,javascript,138625,17621,138625,138625,127859
3,php,578118,21363,578118,578118,483052
4,python,457461,13590,457461,457461,434527
5,ruby,53279,6329,53279,53279,51111



Exact duplicate summary


,language,duplicate_type,duplicate_groups,records_in_duplicate_groups,removable_duplicate_records,largest_group
0,go,exact_comment,18346,86627,68281,2658
1,java,exact_comment,21648,94502,72854,14304
2,javascript,exact_comment,5583,16349,10766,151
3,php,exact_comment,22851,117917,95066,37757
4,python,exact_comment,13383,36317,22934,1779
5,ruby,exact_comment,1446,3614,2168,112



Duplicates crossing original dataset splits


,language,duplicate_type,cross_split_groups,records_in_cross_split_groups,largest_cross_split_group
0,go,exact_comment,560,9043,2658
1,java,exact_comment,780,20751,14304
2,javascript,exact_comment,1083,4338,151
3,php,exact_comment,1813,56956,37757
4,python,exact_comment,640,1831,42
5,ruby,exact_comment,67,405,112



Largest exact-pair duplicate groups


,language,exact_pair_hash,records,repositories,splits



DUPLICATE AND REPOSITORY AUDIT COMPLETED


In [13]:
connection = sqlite3.connect(
    CANDIDATE_DB_PATH,
    timeout=120,
)

try:
    repository_split_membership = pd.read_sql_query("""
        SELECT
            language,
            repository,
            COUNT(*) AS records,
            MAX(original_split = 'train') AS in_train,
            MAX(original_split = 'valid') AS in_valid,
            MAX(original_split = 'test') AS in_test,
            COUNT(DISTINCT original_split) AS split_count
        FROM candidate_records
        GROUP BY language, repository
    """, connection)

finally:
    connection.close()


repository_overlap_summary = (
    repository_split_membership
    .groupby("language", as_index=False)
    .agg(
        repositories=("repository", "count"),
        records=("records", "sum"),
        multi_split_repositories=(
            "split_count",
            lambda values: int((values > 1).sum()),
        ),
        records_in_multi_split_repositories=(
            "records",
            lambda values: int(
                values[
                    repository_split_membership.loc[
                        values.index,
                        "split_count",
                    ] > 1
                ].sum()
            ),
        ),
    )
)

repository_overlap_summary[
    "multi_split_repository_percent"
] = (
    100
    * repository_overlap_summary["multi_split_repositories"]
    / repository_overlap_summary["repositories"]
).round(2)

repository_overlap_summary[
    "records_in_multi_split_repository_percent"
] = (
    100
    * repository_overlap_summary[
        "records_in_multi_split_repositories"
    ]
    / repository_overlap_summary["records"]
).round(2)


pairwise_overlap_rows = []

for language in LANGUAGES:
    language_repositories = repository_split_membership.loc[
        repository_split_membership["language"] == language
    ]

    pairwise_overlap_rows.append({
        "language": language,
        "train_and_valid": int(
            (
                (language_repositories["in_train"] == 1)
                & (language_repositories["in_valid"] == 1)
            ).sum()
        ),
        "train_and_test": int(
            (
                (language_repositories["in_train"] == 1)
                & (language_repositories["in_test"] == 1)
            ).sum()
        ),
        "valid_and_test": int(
            (
                (language_repositories["in_valid"] == 1)
                & (language_repositories["in_test"] == 1)
            ).sum()
        ),
        "all_three_splits": int(
            (
                (language_repositories["in_train"] == 1)
                & (language_repositories["in_valid"] == 1)
                & (language_repositories["in_test"] == 1)
            ).sum()
        ),
    })

pairwise_repository_overlap = pd.DataFrame(
    pairwise_overlap_rows
)


repository_size_summary = (
    repository_split_membership
    .groupby("language")["records"]
    .agg(
        repositories="count",
        minimum="min",
        median="median",
        mean="mean",
        maximum="max",
    )
    .reset_index()
)

repository_size_summary["p90"] = (
    repository_split_membership
    .groupby("language")["records"]
    .quantile(0.90)
    .values
)

repository_size_summary["p95"] = (
    repository_split_membership
    .groupby("language")["records"]
    .quantile(0.95)
    .values
)

repository_size_summary["p99"] = (
    repository_split_membership
    .groupby("language")["records"]
    .quantile(0.99)
    .values
)

repository_size_summary[
    ["mean", "median", "p90", "p95", "p99"]
] = repository_size_summary[
    ["mean", "median", "p90", "p95", "p99"]
].round(2)


largest_repositories = (
    repository_split_membership
    .sort_values(
        ["language", "records"],
        ascending=[True, False],
    )
    .groupby("language", as_index=False)
    .head(5)
    .reset_index(drop=True)
)


print("Repository overlap across original splits")
display(repository_overlap_summary)

print("\nPairwise repository overlap")
display(pairwise_repository_overlap)

print("\nRepository-size distribution")
display(repository_size_summary)

print("\nFive largest repositories per language")
display(largest_repositories)

assert repository_overlap_summary["records"].sum() == 2_070_536
assert (
    repository_overlap_summary["repositories"].sum()
    == len(repository_split_membership)
)

print("\nREPOSITORY SPLIT AUDIT COMPLETED")

Repository overlap across original splits


,language,repositories,records,multi_split_repositories,records_in_multi_split_repositories,multi_split_repository_percent,records_in_multi_split_repository_percent
0,go,4246,346365,0,0,0.0,0.0
1,java,4769,496688,0,0,0.0,0.0
2,javascript,17621,138625,0,0,0.0,0.0
3,php,21363,578118,0,0,0.0,0.0
4,python,13590,457461,0,0,0.0,0.0
5,ruby,6329,53279,0,0,0.0,0.0



Pairwise repository overlap


,language,train_and_valid,train_and_test,valid_and_test,all_three_splits
0,go,0,0,0,0
1,java,0,0,0,0
2,javascript,0,0,0,0
3,php,0,0,0,0
4,python,0,0,0,0
5,ruby,0,0,0,0



Repository-size distribution


,language,repositories,minimum,median,mean,maximum,p90,p95,p99
0,go,4246,1,12.0,81.57,60041,113.0,239.75,957.05
1,java,4769,1,20.0,104.15,22028,194.0,369.00,1294.56
2,javascript,17621,1,2.0,7.87,2049,14.0,27.00,93.00
3,php,21363,1,8.0,27.06,16193,50.0,88.00,320.38
4,python,13590,1,10.0,33.66,11211,70.0,127.00,372.33
5,ruby,6329,1,3.0,8.42,5418,16.0,27.00,76.44



Five largest repositories per language


,language,repository,records,in_train,in_valid,in_test,split_count
0,go,aws/aws-sdk-go,60041,1,0,0,1
1,go,kubernetes/kubernetes,13137,1,0,0,1
2,go,juju/juju,9043,1,0,0,1
3,go,keybase/client,4490,1,0,0,1
4,go,luci/luci-go,4420,1,0,0,1
5,java,aws/aws-sdk-java,22028,1,0,0,1
6,java,OpenLiberty/open-liberty,19469,1,0,0,1
7,java,alkacon/opencms-core,10507,1,0,0,1
8,java,Azure/azure-sdk-for-java,10240,1,0,0,1
9,java,google/j2objc,8046,1,0,0,1



REPOSITORY SPLIT AUDIT COMPLETED


In [15]:
import math
import sqlite3
from statistics import NormalDist

CANDIDATE_SAMPLE_SIZES = (
    250,
    500,
    750,
    1_000,
    1_500,
    2_000,
    3_000,
    4_000,
)

ALPHA = 0.05
BONFERRONI_COMPARISONS = len(LANGUAGES)
POWER_LEVELS = (0.80, 0.90)

connection = sqlite3.connect(
    CANDIDATE_DB_PATH,
    timeout=120,
)

try:
    repository_population = pd.read_sql_query("""
        SELECT
            language,
            COUNT(DISTINCT repository)
                AS population_repositories
        FROM candidate_records
        GROUP BY language
        ORDER BY language
    """, connection)

finally:
    connection.close()


minimum_repository_population = int(
    repository_population[
        "population_repositories"
    ].min()
)

assert max(CANDIDATE_SAMPLE_SIZES) <= (
    minimum_repository_population
), (
    "At least one candidate sample size exceeds "
    "the smallest repository population."
)


def critical_z(alpha):
    return NormalDist().inv_cdf(
        1 - alpha / 2
    )


def power_z(power):
    return NormalDist().inv_cdf(power)


def finite_population_margin(
    population_size,
    sample_size,
    alpha,
):
    z_value = critical_z(alpha)

    standard_error = math.sqrt(
        0.25 / sample_size
    )

    finite_population_correction = math.sqrt(
        (population_size - sample_size)
        / (population_size - 1)
    )

    return (
        z_value
        * standard_error
        * finite_population_correction
    )


def minimum_detectable_correlation(
    sample_size,
    alpha,
    power,
):
    z_alpha = critical_z(alpha)
    z_power = power_z(power)

    fisher_z_effect = (
        z_alpha + z_power
    ) / math.sqrt(sample_size - 3)

    return math.tanh(fisher_z_effect)


def approximate_minimum_detectable_paired_effect(
    sample_size,
    alpha,
    power,
):
    z_alpha = critical_z(alpha)
    z_power = power_z(power)

    return (
        z_alpha + z_power
    ) / math.sqrt(sample_size)


precision_rows = []

for row in repository_population.itertuples(
    index=False
):
    population_size = int(
        row.population_repositories
    )

    for sample_size in CANDIDATE_SAMPLE_SIZES:
        precision_rows.append({
            "language": row.language,
            "population_repositories": population_size,
            "sample_size": sample_size,
            "repository_coverage_percent": round(
                100
                * sample_size
                / population_size,
                2,
            ),
            "worst_case_margin_percent": round(
                100
                * finite_population_margin(
                    population_size,
                    sample_size,
                    ALPHA,
                ),
                3,
            ),
        })


language_precision_audit = pd.DataFrame(
    precision_rows
)


design_rows = []

for sample_size in CANDIDATE_SAMPLE_SIZES:
    candidate_precision = (
        language_precision_audit.loc[
            language_precision_audit[
                "sample_size"
            ] == sample_size
        ]
    )

    design_rows.append({
        "sample_size_per_language": sample_size,
        "total_balanced_sample": (
            sample_size * len(LANGUAGES)
        ),
        "minimum_repository_coverage_percent": (
            candidate_precision[
                "repository_coverage_percent"
            ].min()
        ),
        "maximum_repository_coverage_percent": (
            candidate_precision[
                "repository_coverage_percent"
            ].max()
        ),
        "worst_margin_percent_alpha_0_05": (
            candidate_precision[
                "worst_case_margin_percent"
            ].max()
        ),
        "mde_correlation_power_80": round(
            minimum_detectable_correlation(
                sample_size,
                ALPHA,
                0.80,
            ),
            4,
        ),
        "mde_correlation_power_90": round(
            minimum_detectable_correlation(
                sample_size,
                ALPHA,
                0.90,
            ),
            4,
        ),
        "mde_correlation_power_80_bonferroni": round(
            minimum_detectable_correlation(
                sample_size,
                ALPHA
                / BONFERRONI_COMPARISONS,
                0.80,
            ),
            4,
        ),
        "approx_mde_paired_d_power_80": round(
            approximate_minimum_detectable_paired_effect(
                sample_size,
                ALPHA,
                0.80,
            ),
            4,
        ),
        "approx_mde_paired_d_power_90": round(
            approximate_minimum_detectable_paired_effect(
                sample_size,
                ALPHA,
                0.90,
            ),
            4,
        ),
        "approx_mde_paired_d_power_80_bonferroni": round(
            approximate_minimum_detectable_paired_effect(
                sample_size,
                ALPHA
                / BONFERRONI_COMPARISONS,
                0.80,
            ),
            4,
        ),
    })


sample_size_design_audit = pd.DataFrame(
    design_rows
)


print("Candidate sample-size comparison")
display(sample_size_design_audit)

print(
    "\nWorst-case 95% margin of error "
    "by language and sample size"
)

display(
    language_precision_audit.pivot(
        index="language",
        columns="sample_size",
        values="worst_case_margin_percent",
    )
)

print(
    "\nRepository coverage by language "
    "and sample size"
)

display(
    language_precision_audit.pivot(
        index="language",
        columns="sample_size",
        values="repository_coverage_percent",
    )
)

assert len(sample_size_design_audit) == len(
    CANDIDATE_SAMPLE_SIZES
)

assert len(language_precision_audit) == (
    len(LANGUAGES)
    * len(CANDIDATE_SAMPLE_SIZES)
)

print("\nSAMPLE-SIZE CANDIDATES EVALUATED")
print("No sample size was selected.")
print("No sample records were selected.")
print("No files or database tables were modified.")

Candidate sample-size comparison


,sample_size_per_language,total_balanced_sample,minimum_repository_coverage_percent,maximum_repository_coverage_percent,worst_margin_percent_alpha_0_05,mde_correlation_power_80,mde_correlation_power_90,mde_correlation_power_80_bonferroni,approx_mde_paired_d_power_80,approx_mde_paired_d_power_90,approx_mde_paired_d_power_80_bonferroni
0,250,1500,1.17,5.89,6.162,0.1764,0.2034,0.2179,0.1772,0.2050,0.2201
1,500,3000,2.34,11.78,4.331,0.1250,0.1444,0.1548,0.1253,0.1450,0.1556
2,750,4500,3.51,17.66,3.515,0.1021,0.1180,0.1266,0.1023,0.1184,0.1271
3,1000,6000,4.68,23.55,3.026,0.0885,0.1023,0.1098,0.0886,0.1025,0.1100
4,1500,9000,7.02,35.33,2.440,0.0723,0.0836,0.0897,0.0723,0.0837,0.0899
5,2000,12000,9.36,47.10,2.086,0.0626,0.0724,0.0777,0.0626,0.0725,0.0778
6,3000,18000,14.04,70.65,1.659,0.0511,0.0591,0.0635,0.0511,0.0592,0.0635
7,4000,24000,18.72,94.21,1.397,0.0443,0.0512,0.0550,0.0443,0.0513,0.0550



Worst-case 95% margin of error by language and sample size


sample_size,250,500,750,1000,1500,2000,3000,4000
language,,,,,,,,
go,6.013,4.117,3.247,2.710,2.035,1.594,0.969,0.373
java,6.034,4.147,3.285,2.755,2.095,1.670,1.090,0.622
javascript,6.154,4.320,3.502,3.010,2.420,2.063,1.630,1.362
php,6.162,4.331,3.515,3.026,2.440,2.086,1.659,1.397
python,6.141,4.301,3.478,2.983,2.387,2.024,1.579,1.302
ruby,6.075,4.206,3.360,2.844,2.210,1.812,1.298,0.940



Repository coverage by language and sample size


sample_size,250,500,750,1000,1500,2000,3000,4000
language,,,,,,,,
go,5.89,11.78,17.66,23.55,35.33,47.10,70.65,94.21
java,5.24,10.48,15.73,20.97,31.45,41.94,62.91,83.88
javascript,1.42,2.84,4.26,5.68,8.51,11.35,17.03,22.70
php,1.17,2.34,3.51,4.68,7.02,9.36,14.04,18.72
python,1.84,3.68,5.52,7.36,11.04,14.72,22.08,29.43
ruby,3.95,7.90,11.85,15.80,23.70,31.60,47.40,63.20



SAMPLE-SIZE CANDIDATES EVALUATED
No sample size was selected.
No sample records were selected.
No files or database tables were modified.


In [16]:
connection = sqlite3.connect(
    CANDIDATE_DB_PATH,
    timeout=120,
)

try:
    split_sampling_frame = pd.read_sql_query("""
        SELECT
            language,
            original_split,
            COUNT(*) AS records,
            COUNT(DISTINCT repository) AS repositories
        FROM candidate_records
        GROUP BY language, original_split
        ORDER BY
            language,
            CASE original_split
                WHEN 'train' THEN 1
                WHEN 'valid' THEN 2
                WHEN 'test' THEN 3
            END
    """, connection)

finally:
    connection.close()


test_sampling_frame = (
    split_sampling_frame.loc[
        split_sampling_frame["original_split"] == "test",
        [
            "language",
            "records",
            "repositories",
        ],
    ]
    .rename(columns={
        "records": "test_records",
        "repositories": "test_repositories",
    })
    .reset_index(drop=True)
)


feasibility_rows = []

for row in test_sampling_frame.itertuples(index=False):
    for sample_size in CANDIDATE_SAMPLE_SIZES:
        feasible = sample_size <= row.test_repositories

        feasibility_rows.append({
            "language": row.language,
            "sample_size": sample_size,
            "test_repositories": row.test_repositories,
            "one_per_repository_feasible": feasible,
            "repository_coverage_percent": (
                round(
                    100
                    * sample_size
                    / row.test_repositories,
                    2,
                )
                if feasible
                else None
            ),
        })


test_sample_size_feasibility = pd.DataFrame(
    feasibility_rows
)


print("Sampling frame by original partition")
display(split_sampling_frame)

print("\nTest-partition sampling frame")
display(test_sampling_frame)

print(
    "\nCandidate sizes feasible with "
    "at most one method per test repository"
)

display(
    test_sample_size_feasibility.pivot(
        index="language",
        columns="sample_size",
        values="one_per_repository_feasible",
    )
)

print(
    "\nTest-repository coverage for feasible candidates"
)

display(
    test_sample_size_feasibility.pivot(
        index="language",
        columns="sample_size",
        values="repository_coverage_percent",
    )
)

assert split_sampling_frame["records"].sum() == 2_070_536

assert set(split_sampling_frame["original_split"]) == {
    "train",
    "valid",
    "test",
}

print("\nPARTITION-SPECIFIC SAMPLING FRAME EVALUATED")
print("No sample was selected.")
print("No files or database tables were modified.")

Sampling frame by original partition


,language,original_split,records,repositories
0,go,train,317832,3821
1,go,valid,14242,212
2,go,test,14291,213
3,java,train,454451,4292
4,java,valid,15328,238
5,java,test,26909,239
6,javascript,train,123889,15858
7,javascript,valid,8253,881
8,javascript,test,6483,882
9,php,train,523712,19226



Test-partition sampling frame


,language,test_records,test_repositories
0,go,14291,213
1,java,26909,239
2,javascript,6483,882
3,php,28391,1069
4,python,22176,680
5,ruby,2279,317



Candidate sizes feasible with at most one method per test repository


sample_size,250,500,750,1000,1500,2000,3000,4000
language,,,,,,,,
go,False,False,False,False,False,False,False,False
java,False,False,False,False,False,False,False,False
javascript,True,True,True,False,False,False,False,False
php,True,True,True,True,False,False,False,False
python,True,True,False,False,False,False,False,False
ruby,True,False,False,False,False,False,False,False



Test-repository coverage for feasible candidates


sample_size,250,500,750,1000,1500,2000,3000,4000
language,,,,,,,,
go,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
java,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
javascript,28.34,56.69,85.03,NaN,NaN,NaN,NaN,NaN
php,23.39,46.77,70.16,93.55,NaN,NaN,NaN,NaN
python,36.76,73.53,NaN,NaN,NaN,NaN,NaN,NaN
ruby,78.86,NaN,NaN,NaN,NaN,NaN,NaN,NaN



PARTITION-SPECIFIC SAMPLING FRAME EVALUATED
No sample was selected.
No files or database tables were modified.


In [18]:
import importlib.util
import subprocess
import sys
from importlib.metadata import version

if importlib.util.find_spec("transformers") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "transformers==4.56.2",
    ])

from transformers import RobertaTokenizer

CODEBERT_CHECKPOINT = "microsoft/codebert-base"
CODEBERT_MAX_LENGTH = 512

codebert_tokenizer = RobertaTokenizer.from_pretrained(
    CODEBERT_CHECKPOINT
)

pair_special_token_count = (
    codebert_tokenizer.num_special_tokens_to_add(
        pair=True
    )
)

probe_code = "def add(left, right): return left + right"
probe_comment = "Return the sum of two values."

probe_encoding = codebert_tokenizer(
    probe_code,
    probe_comment,
    add_special_tokens=True,
    truncation=False,
)

probe_tokens = codebert_tokenizer.convert_ids_to_tokens(
    probe_encoding["input_ids"]
)

tokenizer_configuration = pd.DataFrame([{
    "transformers_version": version("transformers"),
    "checkpoint": CODEBERT_CHECKPOINT,
    "tokenizer_class": type(codebert_tokenizer).__name__,
    "model_max_length": codebert_tokenizer.model_max_length,
    "pair_special_tokens": pair_special_token_count,
    "probe_pair_token_count": len(
        probe_encoding["input_ids"]
    ),
}])

print("CodeBERT tokenizer configuration")
display(tokenizer_configuration)

print("\nSynthetic pair token sequence")
print(probe_tokens)

assert (
    codebert_tokenizer.model_max_length
    == CODEBERT_MAX_LENGTH
)

assert pair_special_token_count == 4

assert (
    len(probe_encoding["input_ids"])
    == len(probe_tokens)
)

print("\nCODEBERT TOKENIZER CONFIGURATION VERIFIED")

C:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: c8b9a035-dca6-41bb-ac1f-1f602e585891)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/microsoft/codebert-base/3b0952feddeffad0063f274080e3c23d75e7eb39/vocab.json
Retrying in 1s [Retry 1/5].


CodeBERT tokenizer configuration


,transformers_version,checkpoint,tokenizer_class,model_max_length,pair_special_tokens,probe_pair_token_count
0,4.56.2,microsoft/codebert-base,RobertaTokenizer,512,4,22



Synthetic pair token sequence
['<s>', 'def', 'Ġadd', '(', 'left', ',', 'Ġright', '):', 'Ġreturn', 'Ġleft', 'Ġ+', 'Ġright', '</s>', '</s>', 'Return', 'Ġthe', 'Ġsum', 'Ġof', 'Ġtwo', 'Ġvalues', '.', '</s>']

CODEBERT TOKENIZER CONFIGURATION VERIFIED
